In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession

def extrair_e_carregar_cambio():
    """
    EXTRACT & LOAD (ELT): Busca o JSON bruto da API de câmbio da Argentina 
    e carrega diretamente na camada Bronze do Delta Lake sem transformações.
    """
    url = "https://api.argentinadatos.com/v1/cotizaciones/dolares"
    
    print("🔄 [Extract] Consumindo dados brutos da API...")
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        dados_brutos = response.json()
    except Exception as e:
        raise RuntimeError(f"❌ Falha ao consumir a API: {e}")
        
    if not dados_brutos:
        print("⚠️ Nenhum dado retornado pela API.")
        return
        
    # 1. Cria o DataFrame do Pandas temporário para tipagem inicial
    df_pandas = pd.DataFrame(dados_brutos)
    
    # Garantimos apenas que a data seja string para evitar problemas de fuso horário no Spark
    df_pandas["fecha"] = pd.to_datetime(df_pandas["fecha"]).dt.strftime("%Y-%m-%d")
    
    # 2. Converte para Spark DataFrame (Padrão Databricks)
    df_spark = spark.createDataFrame(df_pandas)
    
    # 3. Cria a tabela 'bronze' no catálogo atual
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
    
    # 4. [Load] Grava o dado bruto na tabela Bronze como Delta Lake
    print("💾 [Load] Salvando dados brutos na tabela bronze.cambio_argentina_raw...")
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze.cambio_argentina_raw")
        
    print("✅ Ingestão Bronze concluída com sucesso!")

# Executa o pipeline
extrair_e_carregar_cambio()

In [0]:
%sql
SELECT * FROM bronze.cambio_argentina_raw order by fecha asc LIMIT 10;